# Classic ML Validation: Data Quality and Drift

In [ ]:
import pandas as pd
from sklearn import datasets

from evidently import Dataset
from evidently import DataDefinition
from evidently import Report
from evidently.presets import DataDriftPreset, DataSummaryPreset

### Let's load tabular dataset for binary classification problem

In [ ]:
adult_data = datasets.fetch_openml(name="adult", version=2, as_frame="auto")
adult = adult_data.frame

In [ ]:
#incase of issues with the data downloading uncomment the code below
# !pip install pyarrow

# url = "https://github.com/evidentlyai/evidently/blob/main/test_data/adults.parquet?raw=true"
# adult = pd.read_parquet(url, engine='pyarrow')
# adult.head()

In [ ]:
adult_ref = adult[~adult.education.isin(["Some-college", "HS-grad", "Bachelors"])] #reference dataset - a basis for comparison
adult_prod = adult[adult.education.isin(["Some-college", "HS-grad", "Bachelors"])] #fresh batch of data to compare with the reference one

### Quick report generation with zero configuration

In [ ]:
report = Report([
    DataDriftPreset()
],
                include_tests="True")

drift_eval = report.run(adult_ref, adult_prod)

In [ ]:
drift_eval

In [ ]:
drift_eval.json()

In [ ]:
drift_eval.dict()

### Configured report generation with data definition

In [ ]:
#create a schema for datasets

schema = DataDefinition(
    numerical_columns=["education-num", "age", "capital-gain", "hours-per-week", "capital-loss", "fnlwgt"],
    categorical_columns=["education", "occupation", "native-country", "workclass", "marital-status", "relationship", "race", "sex", "class"],
    )

In [ ]:
#create evidently datasets from a schema and pandas dataframes

evi_prod_data = Dataset.from_pandas(
    pd.DataFrame(adult_prod),
    data_definition=schema
)

evi_ref_data = Dataset.from_pandas(
    pd.DataFrame(adult_ref),
    data_definition=schema
)

In [ ]:
report = Report([
    DataSummaryPreset()
])

summary_eval = report.run(evi_prod_data, evi_ref_data)

In [ ]:
summary_eval